# Graphiti on Databricks serverless

This notebook runs Graphiti directly in Python on serverless compute, stores its embedded Kuzu graph in a Unity Catalog Volume, and records search queries and results in a Delta table.

Kuzu is suitable for this serverless proof of concept but is deprecated upstream. For a long-running or production Graphiti service, use the repository's FalkorDB runtime in a Databricks App and persist snapshots to a Unity Catalog Volume.

In [ ]:
%pip install --upgrade "typing_extensions>=4.15" "graphiti-core[kuzu]==0.30.2"

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_aws_catalog", "Catalog")
dbutils.widgets.text("schema", "ai_search", "Schema")
dbutils.widgets.text("volume", "graphiti_serverless", "Volume")
dbutils.widgets.text("group_id", "serverless-demo", "Graphiti group")
dbutils.widgets.text("chat_model", "databricks-gpt-5-nano", "Chat model")
dbutils.widgets.text("embedding_model", "databricks-gte-large-en", "Embedding model")
dbutils.widgets.text("query", "What does Graphiti remember about serverless persistence?", "Search query")

In [ ]:
import json
import logging
import re
from datetime import datetime, timezone
from pathlib import Path

from databricks.sdk import WorkspaceClient
from graphiti_core import Graphiti
from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient
from graphiti_core.driver.kuzu_driver import KuzuDriver
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.llm_client.openai_generic_client import OpenAIGenericClient
from graphiti_core.nodes import EpisodeType
from openai import AsyncOpenAI

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("graphiti-serverless")


def _identifier(value: str) -> str:
    """Validate a widget value before using it as a SQL identifier."""
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid SQL identifier: {value!r}")
    return value


catalog = _identifier(dbutils.widgets.get("catalog"))
schema = _identifier(dbutils.widgets.get("schema"))
volume = _identifier(dbutils.widgets.get("volume"))
group_id = dbutils.widgets.get("group_id")
chat_model = dbutils.widgets.get("chat_model")
embedding_model = dbutils.widgets.get("embedding_model")
query = dbutils.widgets.get("query")

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume}`")
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`{schema}`.`graphiti_query_history` (
      queried_at TIMESTAMP,
      group_id STRING,
      query STRING,
      result_json STRING
    ) USING DELTA
    """
)

graph_path = Path(f"/Volumes/{catalog}/{schema}/{volume}/kuzu")
graph_path.mkdir(parents=True, exist_ok=True)
logger.info("Using persistent graph path %s", graph_path)

In [ ]:
workspace = WorkspaceClient()
auth_headers = workspace.config.authenticate()
authorization = next(
    value for key, value in auth_headers.items() if key.lower() == "authorization"
)
token = authorization.removeprefix("Bearer ")
base_url = f"{workspace.config.host.rstrip('/')}/serving-endpoints"
openai_client = AsyncOpenAI(api_key=token, base_url=base_url)
llm_config = LLMConfig(
    api_key=token,
    base_url=base_url,
    model=chat_model,
    small_model=chat_model,
    temperature=0,
)
llm = OpenAIGenericClient(
    config=llm_config,
    client=openai_client,
    structured_output_mode="json_object",
)
embedder = OpenAIEmbedder(
    config=OpenAIEmbedderConfig(
        api_key=token,
        base_url=base_url,
        embedding_model=embedding_model,
        embedding_dim=1024,
    ),
    client=openai_client,
)
driver = KuzuDriver(db=str(graph_path), max_concurrent_queries=1)
graphiti = Graphiti(
    graph_driver=driver,
    llm_client=llm,
    embedder=embedder,
    cross_encoder=OpenAIRerankerClient(config=llm_config, client=openai_client),
)
await graphiti.build_indices_and_constraints()
logger.info("Graphiti started with %s and %s", chat_model, embedding_model)

In [ ]:
episode_time = datetime.now(timezone.utc)
episode = await graphiti.add_episode(
    name=f"serverless-persistence-{episode_time.isoformat()}",
    episode_body=(
        "Graphiti is running in a Databricks serverless notebook. "
        "Its Kuzu graph is stored in a Unity Catalog Volume so later notebook sessions "
        "can reopen the same graph. Search queries and results are also saved in Delta."
    ),
    source=EpisodeType.text,
    source_description="Databricks serverless Graphiti persistence test",
    reference_time=episode_time,
    group_id=group_id,
)
logger.info(
    "Added episode %s with %d nodes and %d edges",
    episode.episode.uuid,
    len(episode.nodes),
    len(episode.edges),
)

In [ ]:
results = await graphiti.search(query, group_ids=[group_id])
result_json = json.dumps(
    [result.model_dump(mode="json") for result in results],
    default=str,
)
query_rows = [(datetime.now(timezone.utc), group_id, query, result_json)]
query_schema = "queried_at timestamp, group_id string, query string, result_json string"
(
    spark.createDataFrame(query_rows, query_schema)
    .write.mode("append")
    .saveAsTable(f"{catalog}.{schema}.graphiti_query_history")
)
logger.info("Saved query with %d results", len(results))
display(
    spark.sql(
        f"""
        SELECT queried_at, group_id, query, result_json
        FROM `{catalog}`.`{schema}`.`graphiti_query_history`
        WHERE group_id = '{group_id.replace("'", "''")}'
        ORDER BY queried_at DESC
        LIMIT 20
        """
    )
)

In [ ]:
await graphiti.close()
await openai_client.close()
logger.info("Graphiti resources closed; persisted files remain under %s", graph_path)